In [1]:
import spacy
nlp = spacy.load("en_core_web_sm")
import time
t1 = time.time()
def extract_keywords(query):
    doc = nlp(query)
    keywords = []

    for token in doc:
        # Extract nouns & proper nouns
        if token.pos_ in ["NOUN", "PROPN"]:
            keywords.append(token.text.lower())

    return keywords

print(extract_keywords("I am going for a trip"))
t2 = time.time()
print(f"Time taken: {t2 - t1} seconds")

print(extract_keywords("I want clothes for Himachal"))
t3 = time.time()
print(f"Time taken: {t3 - t2} seconds")

['trip']
Time taken: 0.052887916564941406 seconds
['clothes', 'himachal']
Time taken: 0.008224248886108398 seconds


In [ ]:
import spacy
import time
import re
from symspellpy import SymSpell

# ==========================================
# 1. DATA KNOWLEDGE BASE (Configuration)
# ==========================================

GEO_DB = {
    # India
    "himachal": ["winter", "thermal"],
    "manali": ["snow", "waterproof"],
    "kashmir": ["heavy wool", "pashmina"],
    "goa": ["beach", "resort"],
    "kerala": ["cotton", "humid"],
    "rajasthan": ["lightweight", "vibrant"],
    "jaipur": ["bandhani", "traditional"],
    "mumbai": ["monsoon", "breathable"],
    "delhi": ["trendy", "urban"],
    # International
    "paris": ["chic", "fashion"],
    "london": ["trench coat", "formal"],
    "dubai": ["luxury", "summer"],
    "thailand": ["swimwear", "beach"],
    "bali": ["resort", "tropical"],
    "canada": ["extreme winter", "parka"],
}

CATEGORY_KEYWORDS = {
    # Tops
    "shirt", "shirts", "tshirt", "t-shirt", "tee", "top", "blouse",
    "tunic", "kurta", "kurti", "crop_top",

    # Outerwear
    "hoodie", "sweatshirt", "sweater", "cardigan",
    "blazer", "jacket", "coat", "parka", "windcheater",

    # Bottoms
    "jeans", "denim", "trousers", "pants", "chinos",
    "leggings", "joggers", "shorts", "skirt", "palazzo",

    # Ethnic
    "saree", "sari", "lehenga", "salwar", "suit", "anarkali",

    # Dresses
    "dress", "gown", "maxi", "midi", "mini",

    # Footwear
    "shoes", "sneakers", "boots", "sandals",
    "heels", "flats", "slippers", "loafers",

    # Accessories
    "watch", "bag", "handbag", "backpack", "purse",
    "wallet", "belt", "scarf", "gloves", "socks",
    "sunglasses", "jewellery", "jewelry", "hat", "cap",

    # Generic
    "clothes", "clothing", "outfit", "wear", "apparel"
}


STYLE_KEYWORDS = {
    "formal", "casual", "smart_casual", "ethnic", "party",
    "sporty", "athleisure", "streetwear",
    "boho", "vintage", "chic", "minimalist",
    "classic", "retro", "luxury", "modest"
}


OCCASION_KEYWORDS = {
    "wedding", "office", "gym", "date", "vacation", "college", 
    "yoga", "running", "travel", "festival", "interview", "party"
}

MATERIAL_KEYWORDS = {
    "cotton", "organic_cotton",
    "silk", "wool", "merino",
    "leather", "faux_leather",
    "linen", "velvet",
    "polyester", "nylon",
    "denim", "satin",
    "fleece", "chiffon", "georgette",
    "rayon", "viscose"
}


GENDER_MAP = {
    # Male
    "men": "male", "man": "male", "male": "male",
    "boy": "male", "boys": "male", "mens": "male",

    # Female
    "women": "female", "woman": "female", "female": "female",
    "girl": "female", "girls": "female",
    "ladies": "female", "womens": "female",

    # Kids
    "kid": "kids", "kids": "kids", "child": "kids",
    "children": "kids", "baby": "kids", "infant": "kids",

    # Neutral
    "unisex": "unisex", "all": "unisex"
}


# Regex for Price
PRICE_PATTERNS = [
    (re.compile(r"under\s?₹?\s?(\d+)", re.I), "max"),
    (re.compile(r"below\s?₹?\s?(\d+)", re.I), "max"),
    (re.compile(r"less than\s?₹?\s?(\d+)", re.I), "max"),
    (re.compile(r"above\s?₹?\s?(\d+)", re.I), "min"),
    (re.compile(r"over\s?₹?\s?(\d+)", re.I), "min"),
    (re.compile(r"between\s?₹?\s?(\d+)\s?(?:and|to)\s?₹?\s?(\d+)", re.I), "range")
]

# ==========================================
# 2. INITIALIZATION & DICTIONARY BUILD
# ==========================================

nlp = spacy.load("en_core_web_sm", disable=["ner"]) 
sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)

def build_dictionary():
    """
    Dynamically builds the dictionary using the Knowledge Base defined above.
    """
    # 1. Structural Words (Stopwords + Connectors)
    structural = {
        "under": 100000, "below": 100000, "above": 100000, "between": 100000,
        "for": 100000, "and": 100000, "in": 100000, "with": 100000, 
        "but": 100000, "not": 100000, "black": 50000, "white": 50000,
        "red": 50000, "blue": 50000, "green": 50000
    }
    
    for w, f in structural.items():
        sym_spell.create_dictionary_entry(w, f)

    # 2. Add all Knowledge Base words (Category, Style, Geo, etc.)
    # We assign them high frequency so they are preferred corrections
    all_knowledge_sets = [
        CATEGORY_KEYWORDS, STYLE_KEYWORDS, OCCASION_KEYWORDS, 
        MATERIAL_KEYWORDS, GENDER_MAP.keys(), GEO_DB.keys()
    ]
    
    for kw_set in all_knowledge_sets:
        for word in kw_set:
            sym_spell.create_dictionary_entry(word, 80000)

build_dictionary()

# ==========================================
# 3. HELPER FUNCTIONS
# ==========================================

def clean_query_for_spellcheck(text):
    """
    Protects numbers from being spell-checked.
    """
    words = text.split()
    corrected_words = []
    
    for word in words:
        # If it contains digits (like '5000' or '5000rs'), skip spell check
        if any(char.isdigit() for char in word):
            corrected_words.append(word)
        else:
            # Word-by-word lookup prevents "wear gown" -> "weargown" merging errors
            suggestions = sym_spell.lookup(word, verbosity=2, max_edit_distance=2)
            if suggestions:
                corrected_words.append(suggestions[0].term)
            else:
                corrected_words.append(word)
                
    return " ".join(corrected_words)

def get_geo_expansion(text_tokens):
    """
    Expands cities/countries into weather/vibe keywords.
    """
    geo_keywords = []
    text_set = set(text_tokens)
    for loc, keywords in GEO_DB.items():
        if loc in text_set:
            geo_keywords.extend(keywords[:2]) 
    return list(set(geo_keywords))

# ==========================================
# 4. MAIN PROCESSOR
# ==========================================

def process_search_query(raw_query: str):
    # 1. Safe Spell Check (Preserves Numbers)
    improved_query = clean_query_for_spellcheck(raw_query.lower())
    
    # 2. NLP Parse
    doc = nlp(improved_query)
    
    result = {
        "improved_query": improved_query,
        "include_keywords": [],
        "exclude_keywords": [],
        "price": {"min": None, "max": None},
        "gender": "unisex",
        "style": [],
        "occasion": [],
        "material": [],
        "category": [],
        "family_friendly": False,
        "child_friendly": False
    }

    tokens_text = []

    # 3. Iterative Token Logic
    for token in doc:
        lemma = token.lemma_
        tokens_text.append(lemma)
        
        # --- Negation Detection ---
        is_negated = False
        # Check children (e.g., "no leather")
        for child in token.children:
            if child.dep_ == "neg": is_negated = True; break
        # Check head (e.g., "don't want leather")
        if not is_negated:
            head = token.head
            for child in head.children:
                if child.dep_ == "neg": is_negated = True; break
        
        # --- Categorization ---
        if token.pos_ in ["NOUN", "PROPN", "ADJ"] and not token.is_stop:
            
            # Check against Knowledge Bases
            if lemma in CATEGORY_KEYWORDS:
                target = result["exclude_keywords"] if is_negated else result["category"]
                target.append(lemma)
                if not is_negated: result["include_keywords"].append(lemma)
            
            elif lemma in STYLE_KEYWORDS:
                result["style"].append(lemma)
                
            elif lemma in OCCASION_KEYWORDS:
                result["occasion"].append(lemma)
                
            elif lemma in MATERIAL_KEYWORDS:
                target = result["exclude_keywords"] if is_negated else result["material"]
                target.append(lemma)
                if not is_negated: result["include_keywords"].append(lemma)

            elif lemma in GENDER_MAP:
                if not is_negated: 
                    # Map "kids" or "boy" to specific logic if needed
                    g_val = GENDER_MAP[lemma]
                    if g_val == "kids": result["child_friendly"] = True
                    else: result["gender"] = g_val

            elif lemma == "family":
                result["family_friendly"] = True
                
            else:
                # General keywords (Colors, miscellaneous attributes)
                # Only add if not already handled by Geo or other logic to keep clean
                if lemma not in GEO_DB: 
                    if is_negated: result["exclude_keywords"].append(lemma)
                    else: result["include_keywords"].append(lemma)

    # 4. Geo Expansion
    geo_exp = get_geo_expansion(tokens_text)
    result["include_keywords"].extend(geo_exp)
    
    # 5. Price Extraction (Regex)
    # We use improved_query because the spell checker preserved the numbers
    for pattern, p_type in PRICE_PATTERNS:
        match = pattern.search(improved_query)
        if match:
            if p_type == "max": result["price"]["max"] = int(match.group(1))
            elif p_type == "min": result["price"]["min"] = int(match.group(1))
            elif p_type == "range":
                result["price"]["min"] = int(match.group(1))
                result["price"]["max"] = int(match.group(2))
            break 

    # 6. Cleanup & De-duplication
    result["include_keywords"] = list(set(result["include_keywords"]))
    result["exclude_keywords"] = list(set(result["exclude_keywords"]))
    # Exclude keywords take priority
    result["include_keywords"] = [k for k in result["include_keywords"] if k not in result["exclude_keywords"]]

    return result

# ==========================================
# 5. TESTING
# ==========================================

def run_test(query):
    t1 = time.time()
    res = process_search_query(query)
    t2 = time.time()
    print("-" * 60)
    print(f"Input:    {query}")
    print(f"Improved: {res['improved_query']}")
    print(f"Time:     {(t2-t1)*1000:.2f} ms")
    print(f"Result:   {res}")

# WARMUP
process_search_query("warmup")

# Test 1: The Fix Verified (Number Protection)
run_test("Womens party wear gown under 5000")

# Test 2: Geo Expansion + Material
run_test("Cotton shirt for Kerala trip")

# Test 3: Negation + Dictionary Correction (lether -> leather)
run_test("Pure lether jacket but not black")

# Test 4: Complex (Gender, Geo, Price, Category)
run_test("Mens winter clothes for Manali under 3000")

# Test 5: Child friendly check
run_test("Shoes for kids for running")

# Test 6: International Geo-Expansion + Style
# Expect: 'paris' -> expands to 'fashion', 'chic'. 'dres' -> 'dress'.
run_test("Chic dres for Paris trip")

# Test 7: Price Range + Specific Gender (Kids)
# Expect: Gender='kids'/'male', Price Min=500, Max=1500.
run_test("Boys jeans between 500 and 1500")

# Test 8: Heavy Spelling Errors + Occasion
# Expect: 'silkk'->'silk', 'sare'->'saree', 'weding'->'wedding'.
run_test("Silkk sare for weding")

# Test 9: Category Negation (Intent Disambiguation)
# Expect: Include 'sneakers', Exclude 'boots'.
run_test("I want sneakers but not boots")

# Test 10: "Above" Price Logic + Luxury Context
# Expect: Price Min=20000.
run_test("Luxury watch above 20000")

# Test 11: Multi-Attribute (Color + Material + Gender + Category)
# Expect: Material='denim', Gender='male', Category='jacket', Include='blue'.
run_test("Blue denim jacket for men")

# Test 12: Family Friendly + Occasion
# Expect: family_friendly=True, Occasion='vacation'.
run_test("Matching tshirts for family vacation")

# Test 13: Cultural/Geo Specifics (Jaipur)
# Expect: 'jaipur' -> expands to 'bandhani', 'traditional'.
run_test("Traditional outfit for Jaipur wedding")

# Test 14: Conflicting Negation (Color)
# Expect: 'red' should be in exclude_keywords, NOT include_keywords.
run_test("Red shirt actually not red")

# Test 15: Season + Geo (Kashmir)
# Expect: 'kashmir' -> 'pashmina', 'heavy wool'. Season 'winter' detected via 'jacket' context implies cold.
run_test("Warm jacket for Kashmir trip")

------------------------------------------------------------
Input:    Womens party wear gown under 5000
Improved: womens party wear gown under 5000
Time:     3.18 ms
Result:   {'improved_query': 'womens party wear gown under 5000', 'include_keywords': ['gown'], 'exclude_keywords': [], 'price': {'min': None, 'max': 5000}, 'gender': 'female', 'style': ['party'], 'occasion': [], 'material': [], 'category': ['gown'], 'family_friendly': False, 'child_friendly': False}
------------------------------------------------------------
Input:    Cotton shirt for Kerala trip
Improved: cotton shirt for kerala top
Time:     3.73 ms
Result:   {'improved_query': 'cotton shirt for kerala top', 'include_keywords': ['cotton', 'shirt', 'humid'], 'exclude_keywords': [], 'price': {'min': None, 'max': None}, 'gender': 'unisex', 'style': [], 'occasion': [], 'material': ['cotton'], 'category': ['shirt'], 'family_friendly': False, 'child_friendly': False}
---------------------------------------------------------

In [32]:
import spacy
import time
import re
from symspellpy import SymSpell

# ==========================================
# 1. DATA KNOWLEDGE BASE (Configuration)
# ==========================================

GEO_DB = {
    # India
    "himachal": ["winter", "thermal", "cold", "layered"],
    "manali": ["snow", "waterproof", "insulated"],
    "kashmir": ["heavy wool", "pashmina", "cold", "winter"],
    "leh": ["extreme cold", "thermal", "down jacket"],
    "ladakh": ["extreme cold", "windproof", "thermal"],

    "goa": ["beach", "resort", "summer", "lightweight"],
    "kerala": ["cotton", "humid", "breathable"],
    "rajasthan": ["lightweight", "vibrant", "ethnic"],
    "jaipur": ["bandhani", "traditional", "festive"],
    "udaipur": ["royal", "ethnic", "wedding"],

    "mumbai": ["monsoon", "breathable", "humid"],
    "delhi": ["trendy", "urban", "layered"],
    "bangalore": ["casual", "comfortable"],
    "chennai": ["cotton", "summer", "humid"],
    "kolkata": ["traditional", "festive"],

    # International
    "paris": ["chic", "fashion", "formal"],
    "london": ["trench coat", "formal", "rainy"],
    "dubai": ["luxury", "summer", "modest"],
    "thailand": ["swimwear", "beach", "tropical"],
    "bali": ["resort", "tropical", "boho"],
    "maldives": ["resort", "swimwear", "luxury"],
    "canada": ["extreme winter", "parka", "snow"],
    "new york": ["urban", "streetwear"],
    "italy": ["luxury", "fashion", "tailored"]
}

CATEGORY_KEYWORDS = {
    # Tops
    "shirt", "shirts", "tshirt", "t-shirt", "tee", "top", "blouse",
    "tunic", "kurta", "kurti", "crop_top", "tank", "tanktop",

    # Outerwear
    "hoodie", "sweatshirt", "sweater", "cardigan",
    "blazer", "jacket", "coat", "parka", "windcheater",
    "windbreaker", "overcoat",

    # Bottoms
    "jeans", "denim", "trousers", "pants", "chinos",
    "leggings", "joggers", "shorts", "skirt", "palazzo",
    "culottes", "trackpants",

    # Ethnic
    "saree", "sari", "lehenga", "salwar", "suit",
    "anarkali", "kurta_set", "dhoti", "sherwani",

    # Dresses
    "dress", "gown", "maxi", "midi", "mini",
    "frock", "jumpsuit", "romper",

    # Footwear
    "shoes", "sneakers", "boots", "sandals",
    "heels", "flats", "slippers", "loafers",
    "trainers", "flipflops",

    # Accessories
    "watch", "bag", "handbag", "backpack", "purse",
    "wallet", "belt", "scarf", "stole", "shawl",
    "gloves", "socks", "cap", "beanie",
    "sunglasses", "jewellery", "jewelry", "hat",

    # Inner / Others
    "innerwear", "sleepwear", "nightwear",
    "activewear", "sportswear",

    # Generic
    "clothes", "clothing", "outfit", "wear", "apparel"
}

STYLE_KEYWORDS = {
    "formal", "casual", "smart_casual", "ethnic", "party",
    "sporty", "athleisure", "streetwear",
    "boho", "vintage", "chic", "minimalist",
    "classic", "retro", "luxury", "modest",
    "royal", "festive", "traditional",
    "modern", "trendy", "elegant", "bold"
}

OCCASION_KEYWORDS = {
    "wedding", "reception", "engagement",
    "office", "work", "meeting",
    "gym", "workout", "fitness",
    "date", "night out",
    "vacation", "holiday", "travel",
    "college", "school",
    "yoga", "running",
    "festival", "diwali", "eid", "christmas",
    "interview", "party", "celebration",
    "beach", "resort"
}

MATERIAL_KEYWORDS = {
    "cotton", "organic_cotton", "khadi",
    "silk", "raw_silk", "wool", "merino",
    "leather", "faux_leather", "suede",
    "linen", "velvet",
    "polyester", "nylon", "acrylic",
    "denim", "satin",
    "fleece", "chiffon", "georgette",
    "rayon", "viscose", "modal", "spandex"
}

GENDER_MAP = {
    # Male
    "men": "male", "man": "male", "male": "male",
    "boy": "male", "boys": "male", "mens": "male",
    "gentlemen": "male", "gents": "male",

    # Female
    "women": "female", "woman": "female", "female": "female",
    "girl": "female", "girls": "female",
    "ladies": "female", "womens": "female",

    # Kids
    "kid": "kids", "kids": "kids", "child": "kids",
    "children": "kids", "baby": "kids", "infant": "kids",
    "toddler": "kids",

    # Neutral
    "unisex": "unisex", "all": "unisex", "everyone": "unisex"
}


# Regex for Price
PRICE_PATTERNS = [
    (re.compile(r"under\s?₹?\s?(\d+)", re.I), "max"),
    (re.compile(r"below\s?₹?\s?(\d+)", re.I), "max"),
    (re.compile(r"less than\s?₹?\s?(\d+)", re.I), "max"),
    (re.compile(r"above\s?₹?\s?(\d+)", re.I), "min"),
    (re.compile(r"over\s?₹?\s?(\d+)", re.I), "min"),
    (re.compile(r"between\s?₹?\s?(\d+)\s?(?:and|to)\s?₹?\s?(\d+)", re.I), "range")
]

# ==========================================
# 2. INITIALIZATION & DICTIONARY BUILD
# ==========================================

nlp = spacy.load("en_core_web_sm", disable=["ner"]) 
sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)

def build_dictionary():
    """
    Dynamically builds the dictionary using the Knowledge Base defined above.
    """
    # 1. Structural Words (Stopwords + Connectors)
    structural = {
        "under": 100000, "below": 100000, "above": 100000, "between": 100000,
        "for": 100000, "and": 100000, "in": 100000, "with": 100000, 
        "but": 100000, "not": 100000, "black": 50000, "white": 50000,
        "red": 50000, "blue": 50000, "green": 50000
    }
    
    for w, f in structural.items():
        sym_spell.create_dictionary_entry(w, f)

    # 2. Add all Knowledge Base words (Category, Style, Geo, etc.)
    # We assign them high frequency so they are preferred corrections
    all_knowledge_sets = [
        CATEGORY_KEYWORDS, STYLE_KEYWORDS, OCCASION_KEYWORDS, 
        MATERIAL_KEYWORDS, GENDER_MAP.keys(), GEO_DB.keys()
    ]
    
    for kw_set in all_knowledge_sets:
        for word in kw_set:
            sym_spell.create_dictionary_entry(word, 80000)

build_dictionary()

# ==========================================
# 3. HELPER FUNCTIONS
# ==========================================

def clean_query_for_spellcheck(text):
    """
    Protects numbers from being spell-checked.
    """
    words = text.split()
    corrected_words = []
    
    for word in words:
        # If it contains digits (like '5000' or '5000rs'), skip spell check
        if any(char.isdigit() for char in word):
            corrected_words.append(word)
        else:
            # Word-by-word lookup prevents "wear gown" -> "weargown" merging errors
            suggestions = sym_spell.lookup(word, verbosity=2, max_edit_distance=2)
            if suggestions:
                corrected_words.append(suggestions[0].term)
            else:
                corrected_words.append(word)
                
    return " ".join(corrected_words)

def get_geo_expansion(text_tokens):
    """
    Expands cities/countries into weather/vibe keywords.
    """
    geo_keywords = []
    text_set = set(text_tokens)
    for loc, keywords in GEO_DB.items():
        if loc in text_set:
            geo_keywords.extend(keywords[:2]) 
    return list(set(geo_keywords))

def remove_excluded_from_query(query: str, exclude_keywords: list):
    """
    Removes excluded keywords from the improved query safely (token-level).
    """
    doc = nlp(query)
    cleaned_tokens = []

    exclude_set = set(exclude_keywords)

    for token in doc:
        # Keep token if its lemma is NOT excluded
        if token.lemma_ not in exclude_set:
            cleaned_tokens.append(token.text)

    return " ".join(cleaned_tokens)


# ==========================================
# 4. MAIN PROCESSOR
# ==========================================

def process_search_query(raw_query: str):
    improved_query = clean_query_for_spellcheck(raw_query.lower())
    doc = nlp(improved_query)

    result = {
        "suggested_query": improved_query,
        "improved_query": improved_query,
        "include_keywords": [],
        "exclude_keywords": [],
        "price": {"min": None, "max": None},
        "gender": "unisex",
        "style": [],
        "occasion": [],
        "material": [],
        "category": [],
        "family_friendly": False,
        "child_friendly": False
    }

    tokens = []
    detected_genders = set()   # 🔧 CHANGED

    for token in doc:
        lemma = token.lemma_
        tokens.append(lemma)

        is_negated = any(c.dep_ == "neg" for c in token.children) or \
                     any(c.dep_ == "neg" for c in token.head.children)

        if token.pos_ in ["NOUN", "ADJ", "PROPN"] and not token.is_stop:

            if lemma in CATEGORY_KEYWORDS:
                (result["exclude_keywords"] if is_negated else result["category"]).append(lemma)
                if not is_negated:
                    result["include_keywords"].append(lemma)

            elif lemma in STYLE_KEYWORDS:
                result["style"].append(lemma)
                result["include_keywords"].append(lemma)

            elif lemma in OCCASION_KEYWORDS:
                result["occasion"].append(lemma)

            elif lemma in MATERIAL_KEYWORDS:
                (result["exclude_keywords"] if is_negated else result["material"]).append(lemma)
                if not is_negated:
                    result["include_keywords"].append(lemma)

            # 🔧 CHANGED GENDER LOGIC
            elif lemma in GENDER_MAP and not is_negated:
                detected_genders.add(GENDER_MAP[lemma])

            elif lemma == "family":
                result["family_friendly"] = True

            else:
                (result["exclude_keywords"] if is_negated else result["include_keywords"]).append(lemma)

    # 🔧 FINAL GENDER RESOLUTION (SAFE)
    if "kids" in detected_genders:
        result["gender"] = "kids"
        result["child_friendly"] = True
    elif "male" in detected_genders and "female" in detected_genders:
        result["gender"] = "unisex"
    elif "male" in detected_genders:
        result["gender"] = "male"
    elif "female" in detected_genders:
        result["gender"] = "female"

    # Geo expansion
    result["include_keywords"].extend(get_geo_expansion(tokens))

    # Price
    for p, t in PRICE_PATTERNS:
        m = p.search(improved_query)
        if m:
            if t == "max":
                result["price"]["max"] = int(m.group(1))
            elif t == "min":
                result["price"]["min"] = int(m.group(1))
            else:
                result["price"]["min"] = int(m.group(1))
                result["price"]["max"] = int(m.group(2))
            break

    result["include_keywords"] = list(set(result["include_keywords"]) - set(result["exclude_keywords"]))
    result["exclude_keywords"] = list(set(result["exclude_keywords"]))

    # 🔧 REMOVE EXCLUDED WORDS FROM QUERY
    if result["exclude_keywords"]:

        result["improved_query"] = remove_excluded_from_query(
            result["improved_query"],
            result["exclude_keywords"]
        )

    return result


# ==========================================
# 5. TESTING
# ==========================================

def run_test(query):
    t1 = time.time()
    res = process_search_query(query)
    t2 = time.time()
    print("-" * 60)
    print(f"Input:    {query}")
    print(f"Improved: {res['improved_query']}")
    print(f"Time:     {(t2-t1)*1000:.2f} ms")
    print(f"Result:   {res}")

# WARMUP
process_search_query("warmup")

# Test 1: The Fix Verified (Number Protection)
run_test("Womens party wear gown under 5000")

# Test 2: Geo Expansion + Material
run_test("Cotton shirt for Kerala trip")

# Test 3: Negation + Dictionary Correction (lether -> leather)
run_test("Pure lether jacket but not black")

# Test 4: Complex (Gender, Geo, Price, Category)
run_test("Mens winter clothes for Manali under 3000")

# Test 5: Child friendly check
run_test("Shoes for kids for running")

# Test 6: International Geo-Expansion + Style
# Expect: 'paris' -> expands to 'fashion', 'chic'. 'dres' -> 'dress'.
run_test("Chic dres for Paris trip")

# Test 7: Price Range + Specific Gender (Kids)
# Expect: Gender='kids'/'male', Price Min=500, Max=1500.
run_test("Boys jeans between 500 and 1500")

# Test 8: Heavy Spelling Errors + Occasion
# Expect: 'silkk'->'silk', 'sare'->'saree', 'weding'->'wedding'.
run_test("Silkk sare for weding")

# Test 9: Category Negation (Intent Disambiguation)
# Expect: Include 'sneakers', Exclude 'boots'.
run_test("I want sneakers but not boots")

# Test 10: "Above" Price Logic + Luxury Context
# Expect: Price Min=20000.
run_test("Luxury watch above 20000")

# Test 11: Multi-Attribute (Color + Material + Gender + Category)
# Expect: Material='denim', Gender='male', Category='jacket', Include='blue'.
run_test("Blue denim jacket for men")

# Test 12: Family Friendly + Occasion
# Expect: family_friendly=True, Occasion='vacation'.
run_test("Matching tshirts for family vacation")

# Test 13: Cultural/Geo Specifics (Jaipur)
# Expect: 'jaipur' -> expands to 'bandhani', 'traditional'.
run_test("Traditional outfit for Jaipur wedding")

# Test 14: Conflicting Negation (Color)
# Expect: 'red' should be in exclude_keywords, NOT include_keywords.
run_test("Red shirt actually not red")

# Test 15: Season + Geo (Kashmir)
# Expect: 'kashmir' -> 'pashmina', 'heavy wool'. Season 'winter' detected via 'jacket' context implies cold.
run_test("Warm jacket for Kashmir trip")

# Test 16: Substring safety
# Expect: gender = 'female' (NOT male)
run_test("Women jackets for winter")

# Test 17: Both genders mentioned
# Expect: gender = 'unisex'
run_test("Furniture for both men and women")

# Test 19: Gender negation
# Expect: gender = 'female'
run_test("Dresses not for men but for women")


# Test 20: Possessive usage
# Expect: gender = 'male'
run_test("Men jackets for office wear")


# Test 21: Explicit unisex
# Expect: gender = 'unisex'
run_test("Unisex sneakers for daily wear")

# Test 22: Geo + gender
# Expect: gender = 'female', include 'bandhani'
run_test("Traditional saree for women in Jaipur")

# Test 23: Male + female + kids
# Expect: gender = 'kids', child_friendly = True
run_test("Tshirts for men women and kids")

# Test 24: Negation with geo expansion
# Expect: exclude 'leather', include 'thermal'
run_test("Not leather jacket for Manali trip")

# Test 25: International city inference
# Expect: include 'chic', 'fashion'
run_test("Chic dress for Paris vacation")

# Test 26: No gender mentioned
# Expect: gender = 'unisex'
run_test("Running shoes under 4000")

# Test 27: Color negation
# Expect: 'black' in exclude_keywords only
run_test("Black jacket not black actually")

# Test 28: Cold weather material inference
# Expect: include 'wool', 'winter'
run_test("Wool coat for Canada trip")

# Test 29: Family intent
# Expect: family_friendly = True
run_test("Matching outfits for family vacation")

# Test 30: Spell correction stress test
# Expect: silk, saree, wedding
run_test("Silkk sare for weding")

------------------------------------------------------------
Input:    Womens party wear gown under 5000
Improved: womens party wear gown under 5000
Time:     3.26 ms
Result:   {'suggested_query': 'womens party wear gown under 5000', 'improved_query': 'womens party wear gown under 5000', 'include_keywords': ['gown', 'party'], 'exclude_keywords': [], 'price': {'min': None, 'max': 5000}, 'gender': 'female', 'style': ['party'], 'occasion': [], 'material': [], 'category': ['gown'], 'family_friendly': False, 'child_friendly': False}
------------------------------------------------------------
Input:    Cotton shirt for Kerala trip
Improved: cotton shirt for kerala top
Time:     3.59 ms
Result:   {'suggested_query': 'cotton shirt for kerala top', 'improved_query': 'cotton shirt for kerala top', 'include_keywords': ['kerala', 'cotton', 'shirt', 'humid'], 'exclude_keywords': [], 'price': {'min': None, 'max': None}, 'gender': 'unisex', 'style': [], 'occasion': [], 'material': ['cotton'], 'categ